# counseLLM — Data Exploration

Explore and visualize the SFT and DPO datasets before training.

**SFT Datasets:**
1. `ShenLab/MentalChat16K` — 16K synthetic + interview
2. `Estwld/empathetic_dialogues_llm` — 24.8K real human multi-turn
3. `EmoCareAI/Psych8k` — ~8K real therapist (gated)
4. `nbertagnolli/counsel-chat` — 2.8K real therapist
5. `thu-coai/esconv` — 1.3K real human + strategy labels

**DPO Dataset:**
- `Psychotherapy-LLM/PsychoCounsel-Preference` — 36.7K preference pairs

In [ ]:
!pip install datasets matplotlib seaborn wordcloud -q

In [ ]:
import json
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from wordcloud import WordCloud

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. MentalChat16K

In [ ]:
mc16k = load_dataset("ShenLab/MentalChat16K", split="train")
print(f"Size: {len(mc16k)}")
print(f"Columns: {mc16k.column_names}")
print(f"\nSample:")
print(f"  Instruction: {mc16k[0]['instruction'][:100]}...")
print(f"  Input: {mc16k[0]['input'][:200]}...")
print(f"  Output: {mc16k[0]['output'][:200]}...")

In [ ]:
# Response length distribution
mc16k_input_lens = [len(row["input"].split()) for row in mc16k]
mc16k_output_lens = [len(row["output"].split()) for row in mc16k]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(mc16k_input_lens, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("MentalChat16K — Input Length (words)")
axes[0].set_xlabel("Word count")
axes[1].hist(mc16k_output_lens, bins=50, color="coral", edgecolor="white")
axes[1].set_title("MentalChat16K — Output Length (words)")
axes[1].set_xlabel("Word count")
plt.tight_layout()
plt.show()

print(f"Input  — Mean: {sum(mc16k_input_lens)/len(mc16k_input_lens):.0f}, "
      f"Min: {min(mc16k_input_lens)}, Max: {max(mc16k_input_lens)}")
print(f"Output — Mean: {sum(mc16k_output_lens)/len(mc16k_output_lens):.0f}, "
      f"Min: {min(mc16k_output_lens)}, Max: {max(mc16k_output_lens)}")

## 2. Empathetic Dialogues

In [ ]:
ed = load_dataset("Estwld/empathetic_dialogues_llm", split="train")
print(f"Size: {len(ed)}")
print(f"Columns: {ed.column_names}")

# Emotion distribution
emotions = [row["emotion"] for row in ed]
emotion_counts = Counter(emotions)
print(f"\nUnique emotions: {len(emotion_counts)}")

# Plot top 20 emotions
top_emotions = emotion_counts.most_common(20)
labels, counts = zip(*top_emotions)
plt.figure(figsize=(14, 6))
plt.barh(list(reversed(labels)), list(reversed(counts)), color="mediumpurple")
plt.title("Empathetic Dialogues — Top 20 Emotion Labels")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Conversation length distribution (number of turns)
ed_turn_counts = [len(row["conversations"]) for row in ed]

plt.figure(figsize=(10, 5))
plt.hist(ed_turn_counts, bins=range(1, max(ed_turn_counts)+2), color="mediumpurple", edgecolor="white")
plt.title("Empathetic Dialogues — Turns per Conversation")
plt.xlabel("Number of turns")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Turns — Mean: {sum(ed_turn_counts)/len(ed_turn_counts):.1f}, "
      f"Min: {min(ed_turn_counts)}, Max: {max(ed_turn_counts)}")

## 3. Counsel Chat

In [ ]:
cc = load_dataset("nbertagnolli/counsel-chat", split="train")
print(f"Size: {len(cc)}")
print(f"Columns: {cc.column_names}")

# Unique questions
unique_questions = len(set(row["questionID"] for row in cc))
print(f"Unique questions: {unique_questions}")
print(f"Avg answers per question: {len(cc)/unique_questions:.1f}")

# Topic distribution
topics = [row["topic"] for row in cc]
topic_counts = Counter(topics)

plt.figure(figsize=(14, 6))
labels, counts = zip(*topic_counts.most_common())
plt.barh(list(reversed(labels)), list(reversed(counts)), color="seagreen")
plt.title("Counsel Chat — Topic Distribution")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Upvote distribution and deduplication preview
upvotes = [row["upvotes"] for row in cc]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(upvotes, bins=range(0, max(upvotes)+2), color="seagreen", edgecolor="white")
axes[0].set_title("Counsel Chat — Upvote Distribution")
axes[0].set_xlabel("Upvotes")

# After dedup: pick highest-upvoted answer per question
question_best = {}
for row in cc:
    qid = row["questionID"]
    if qid not in question_best or row["upvotes"] > question_best[qid]["upvotes"]:
        question_best[qid] = row

deduped_lens = [len(row["answerText"].split()) for row in question_best.values()]
axes[1].hist(deduped_lens, bins=50, color="darkgreen", edgecolor="white")
axes[1].set_title(f"After Dedup ({len(question_best)} examples) — Answer Length")
axes[1].set_xlabel("Word count")
plt.tight_layout()
plt.show()

print(f"Before dedup: {len(cc)} rows | After dedup: {len(question_best)} unique questions")

## 4. ESConv

In [ ]:
esconv = load_dataset("thu-coai/esconv", split="train")
print(f"Size: {len(esconv)}")
print(f"Columns: {esconv.column_names}")

# Parse JSON text column
parsed = []
for row in esconv:
    try:
        data = json.loads(row["text"]) if isinstance(row["text"], str) else row["text"]
        parsed.append(data)
    except (json.JSONDecodeError, TypeError):
        continue

print(f"Successfully parsed: {len(parsed)}")

# Emotion type distribution
emotion_types = [d.get("emotion_type", "unknown") for d in parsed]
print(f"\nEmotion types: {Counter(emotion_types).most_common(10)}")

# Strategy distribution across all supporter turns
strategies = []
for d in parsed:
    for turn in d.get("dialog", []):
        if turn.get("speaker") == "sys" and "strategy" in turn:
            strategies.append(turn["strategy"])

strategy_counts = Counter(strategies)
print(f"\nTherapeutic strategies used:")
labels, counts = zip(*strategy_counts.most_common())
plt.figure(figsize=(12, 5))
plt.barh(list(reversed(labels)), list(reversed(counts)), color="darkorange")
plt.title("ESConv — Therapeutic Strategy Distribution")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

## 5. DPO Dataset — PsychoCounsel-Preference

In [ ]:
pcp = load_dataset("Psychotherapy-LLM/PsychoCounsel-Preference", split="train")
print(f"Size: {len(pcp)}")
print(f"Columns: {pcp.column_names}")

# Unique questions
unique_ids = len(set(row["ID"] for row in pcp))
print(f"Unique question IDs: {unique_ids}")
print(f"Avg pairs per question: {len(pcp)/unique_ids:.1f}")

# Model distribution
chosen_models = Counter(row["chosen_model"] for row in pcp)
rejected_models = Counter(row["rejected_model"] for row in pcp)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
labels_c, counts_c = zip(*chosen_models.most_common())
axes[0].barh(list(reversed(labels_c)), list(reversed(counts_c)), color="forestgreen")
axes[0].set_title("Chosen Model Distribution")

labels_r, counts_r = zip(*rejected_models.most_common())
axes[1].barh(list(reversed(labels_r)), list(reversed(counts_r)), color="indianred")
axes[1].set_title("Rejected Model Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Rating gap analysis
DIMS = ["empathy", "relevance", "clarity", "safety", "exploration", "autonomy", "staging"]

rating_gaps = []
for row in pcp:
    chosen_sum = sum(row.get(f"chosen_{d}_rating", 0) for d in DIMS)
    rejected_sum = sum(row.get(f"rejected_{d}_rating", 0) for d in DIMS)
    rating_gaps.append(chosen_sum - rejected_sum)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(rating_gaps, bins=60, color="teal", edgecolor="white")
axes[0].set_title("Rating Gap Distribution (chosen - rejected)")
axes[0].set_xlabel("Gap (sum of 7 dimensions)")
axes[0].axvline(x=0, color="red", linestyle="--", alpha=0.7)

# Per-dimension average ratings
chosen_avgs = {d: sum(row[f"chosen_{d}_rating"] for row in pcp) / len(pcp) for d in DIMS}
rejected_avgs = {d: sum(row[f"rejected_{d}_rating"] for row in pcp) / len(pcp) for d in DIMS}

x = range(len(DIMS))
width = 0.35
axes[1].bar([i - width/2 for i in x], [chosen_avgs[d] for d in DIMS], width, label="Chosen", color="forestgreen")
axes[1].bar([i + width/2 for i in x], [rejected_avgs[d] for d in DIMS], width, label="Rejected", color="indianred")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(DIMS, rotation=45, ha="right")
axes[1].set_title("Average Ratings by Dimension")
axes[1].set_ylabel("Rating (1-5)")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Rating gap — Mean: {sum(rating_gaps)/len(rating_gaps):.1f}, "
      f"Min: {min(rating_gaps)}, Max: {max(rating_gaps)}")
print(f"Pairs with gap > 5: {sum(1 for g in rating_gaps if g > 5)} "
      f"({sum(1 for g in rating_gaps if g > 5)/len(rating_gaps)*100:.1f}%)")

## 6. Processed Data Summary

Run the data preparation scripts first, then inspect the output.

In [ ]:
# Load processed SFT data (run prepare_sft_data.py first)
import os

sft_meta_path = "../data/sft_metadata.json"
if os.path.exists(sft_meta_path):
    with open(sft_meta_path) as f:
        sft_meta = json.load(f)
    
    print("SFT Dataset Summary")
    print(f"  Total: {sft_meta['total_examples']}")
    print(f"  Train: {sft_meta['train_examples']}")
    print(f"  Val:   {sft_meta['val_examples']}")
    print(f"\nSource distribution:")
    
    sources = sft_meta["source_distribution"]
    labels = list(sources.keys())
    values = list(sources.values())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(labels, values, color=["steelblue", "mediumpurple", "seagreen", "darkorange", "coral"])
    axes[0].set_title("SFT — Examples per Source")
    axes[0].set_xlabel("Count")
    
    axes[1].pie(values, labels=labels, autopct="%1.1f%%",
                colors=["steelblue", "mediumpurple", "seagreen", "darkorange", "coral"])
    axes[1].set_title("SFT — Source Proportions")
    plt.tight_layout()
    plt.show()
else:
    print("Run `python data/prepare_sft_data.py` first to generate processed data.")

In [ ]:
# Inspect a few processed SFT examples
sft_train_path = "../data/sft_train.jsonl"
if os.path.exists(sft_train_path):
    examples = []
    with open(sft_train_path) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            examples.append(json.loads(line))
    
    for i, ex in enumerate(examples):
        msgs = ex["messages"]
        print(f"\n{'='*60}")
        print(f"Example {i+1} — {len(msgs)} messages")
        print(f"{'='*60}")
        for msg in msgs:
            role = msg["role"].upper()
            content = msg["content"][:200]
            print(f"  [{role}] {content}{'...' if len(msg['content']) > 200 else ''}")
else:
    print("Run `python data/prepare_sft_data.py` first.")